In [1]:
import os, json, time

In [2]:
from openai import OpenAI
import utils

key_file = 'openai-api-key.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = OpenAI(  
  api_key=API_KEY
)


In [ ]:
from importlib import reload
reload(utils)

In [3]:
model_name =  'o3-mini'
model_endpoint = utils.model_names_to_endpoints[model_name]
model_endpoint

'o3-mini-2025-01-31'

In [4]:
data_dir = '../data/final_dataset'
short_ans_files = [
    #'certamen_short_answer.json', 
    'junior_scholarship_short_answer.json'
    ]
short_ans_files = [os.path.join(data_dir, f) for f in short_ans_files]

file_to_data = {}
for file in short_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

junior_scholarship_short_answer.json 675


In [5]:
def construct_short_ans_one_word_user_prompt(q_dict):
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    question_text += '\n' + utils.short_ans_one_word_format_instructions

    return question_text

In [6]:
prompt = construct_short_ans_one_word_user_prompt(file_to_data['certamen_short_answer.json'][0])
prompt

KeyError: 'certamen_short_answer.json'

In [7]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [8]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    if os.path.exists(save_file):
        with open(save_file, 'r') as f:
            q_id_to_resp = json.load(f)
        
    for q_dict in data:
        q_id = q_dict['question_id']
        if q_id in q_id_to_resp:
            i += 1
            continue
        prompt = construct_short_ans_one_word_user_prompt(q_dict)

        try:
            response = client.responses.create(
                model=model_endpoint,
                instructions = utils.sys_prompt,
                input = prompt,
            )
            resp = response.output_text
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.01)
        

        if i % 50 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)

junior_scholarship_short_answer.json


  0 / 675
  50 / 675
  100 / 675
  150 / 675
  200 / 675
  250 / 675
Error on latin-grammar-junior-scholarship_35.4.6
Response(id='resp_0beb00c1dd24102a0068faca1db49c81a38fca15fb764cb198', created_at=1761266205.0, error=None, incomplete_details=None, instructions='You are a Classicist with expert knowledge in Greek and Roman history, language, and culture.', metadata={}, model='o3-mini-2025-01-31', object='response', output=[ResponseReasoningItem(id='rs_0beb00c1dd24102a0068faca1ffa3881a3936da8a689d0d75e', summary=[], type='reasoning', encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0beb00c1dd24102a0068faca20d8a881a3b5b6d8c7d3a4f3d4', content=[ResponseOutputText(annotations=[], text='In Latin, the word for "when?" is translated as "quando." \n\nAnswer: quando', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, max_output_token